In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJE     = '/content/drive/MyDrive/cse654_absa'
VERI      = f'{PROJE}/absa_10k_final.csv'
SPLIT_DIR = f'{PROJE}/splits'      # bütün modeller ve LLM deneyi aynı bölmeyi kullanır
SONUC_DIR = f'{PROJE}/sonuclar'
for p in [SPLIT_DIR, SONUC_DIR]:
    os.makedirs(p, exist_ok=True)
assert os.path.exists(VERI), f'CSV bulunamadı: {VERI}'
print('Hazır:', PROJE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hazır: /content/drive/MyDrive/cse654_absa


In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import ast, re, math, inspect
import numpy as np
import pandas as pd
import torch, transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print('transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK, T4 GPU seç!')

SEED = 42
transformers.set_seed(SEED)

# Model HuggingFace'ten otomatik indirilir. İleride ModernBERT-TR / XLM-R için sadece bu iki satırı değiştir.
MODEL_ADI  = 'dbmdz/bert-base-turkish-cased'
MODEL_KISA = 'berturk'
CKPT_DIR   = f'{PROJE}/{MODEL_KISA}_checkpoints'
MODEL_DIR  = f'{PROJE}/{MODEL_KISA}_final'

LABELS   = ['negative', 'neutral', 'positive']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

LR, BATCH, EPOCHS, WARMUP, WD = 2e-5, 32, 4, 0.1, 0.01
RUN_ABLATION = False   # True: target vermeden sadece cümleyle ek eğitim (~15-30 dk)

transformers 5.21.0 | GPU: Tesla T4


In [ ]:
# ---- BÖLME (cümle bazında, 4 grubun her biri %80 eğitim / %20 test; eğitimin %10'u dev) ----
SPLIT_FILE = f'{SPLIT_DIR}/split_ids.csv'

if os.path.exists(SPLIT_FILE):
    kayit = pd.read_csv(SPLIT_FILE)
    if set(kayit.id) == set(df.id) and kayit.split.value_counts(normalize=True).get('test', 0) > 0.15:
        df = df.merge(kayit, on='id', how='left')
        print('Kayıtlı bölme kullanıldı.')
    else:
        os.remove(SPLIT_FILE)
        print('Eski/uyumsuz bölme dosyası silindi, yeniden oluşturulacak.')

if 'split' not in df.columns:
    df['anahtar'] = df.text.str.lower().str.split().str.join(' ')   # tekrar eden cümleler aynı tarafa
    g = df.groupby('anahtar', as_index=False).agg(grup=('grup', 'first'))
    tr_g, te_g  = train_test_split(g,    test_size=0.20, stratify=g.grup,    random_state=SEED)
    tr_g, dev_g = train_test_split(tr_g, test_size=0.10, stratify=tr_g.grup, random_state=SEED)
    m = {**dict.fromkeys(tr_g.anahtar, 'train'), **dict.fromkeys(dev_g.anahtar, 'dev'), **dict.fromkeys(te_g.anahtar, 'test')}
    df['split'] = df.anahtar.map(m)
    df[['id', 'split']].to_csv(SPLIT_FILE, index=False)
    print('Yeni bölme oluşturuldu:', SPLIT_FILE)

assert df['split'].notna().all()
display(pd.crosstab(df.grup, df.split, margins=True)[['train', 'dev', 'test', 'All']])

# ---- CÜMLE–TARGET ÇİFTLERİ ----
rows = []
for r in df.itertuples():
    for k, (t, p) in enumerate(zip(r.targets, r.polarities), start=1):
        rows.append(dict(id=r.id, sira=k, text=r.text, target=t, label=p, category=r.category,
                         grup=r.grup, n_targets=r.n_targets, kelime=r.kelime, baglac=r.baglac, split=r.split))
pairs = pd.DataFrame(rows)
for s in ['train', 'dev', 'test']:
    pairs[pairs.split == s].to_csv(f'{SPLIT_DIR}/{s}_pairs.csv', index=False)   # tuzak test ve LLM için

train_df = pairs[pairs.split == 'train'].reset_index(drop=True)
dev_df   = pairs[pairs.split == 'dev'].reset_index(drop=True)
test_df  = pairs[pairs.split == 'test'].reset_index(drop=True)
display(pd.crosstab(pairs.split, pairs.label, margins=True))

Kayıtlı bölme kullanıldı.
Yeni bölme oluşturuldu: /content/drive/MyDrive/cse654_absa/splits/split_ids.csv


split,train,dev,test,All
grup,,,,
ayni_duygulu,3610,401,1003,5014
notrle_karisik,1607,179,447,2233
tek_hedefli,570,63,159,792
zit_duygulu,1411,157,392,1960
All,7198,800,2001,9999


label,negative,neutral,positive,All
split,,,,
dev,646,512,1010,2168
test,1806,1220,2381,5407
train,6319,4659,8737,19715
All,8771,6391,12128,27290


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_ADI)
lens = np.array([len(x) for x in tok(pairs.text.tolist(), pairs.target.tolist())['input_ids']])
MAX_LEN = int(min(512, math.ceil(np.percentile(lens, 99.9) / 32) * 32))
print(f'Token: ort {lens.mean():.0f}, %99 {np.percentile(lens, 99):.0f}, max {lens.max()} → MAX_LEN={MAX_LEN}')

def hazirla(dfp, with_target=True):
    ds = Dataset.from_pandas(dfp[['text', 'target', 'label']])
    def enc(b):
        out = (tok(b['text'], b['target'], truncation='only_first', max_length=MAX_LEN) if with_target
               else tok(b['text'], truncation=True, max_length=MAX_LEN))
        out['labels'] = [label2id[x] for x in b['label']]
        return out
    return ds.map(enc, batched=True, remove_columns=['text', 'target', 'label'])

train_ds, dev_ds, test_ds = hazirla(train_df), hazirla(dev_df), hazirla(test_df)

Token: ort 31, %99 78, max 103 → MAX_LEN=96


Map:   0%|          | 0/19715 [00:00<?, ? examples/s]

Map:   0%|          | 0/2168 [00:00<?, ? examples/s]

Map:   0%|          | 0/5407 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from transformers.trainer_utils import get_last_checkpoint

def compute_metrics(p):
    y_hat, y = p.predictions.argmax(-1), p.label_ids
    return {'accuracy': accuracy_score(y, y_hat), 'macro_f1': f1_score(y, y_hat, average='macro')}

def egit(train_ds, dev_ds, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ADI, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    kw = dict(output_dir=out_dir, save_strategy='epoch', save_total_limit=2,
              load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
              learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
              num_train_epochs=EPOCHS, warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * EPOCHS),
              weight_decay=WD, fp16=torch.cuda.is_available(), logging_steps=50, report_to='none', seed=SEED)
    kw['eval_strategy' if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters
       else 'evaluation_strategy'] = 'epoch'
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=train_ds, eval_dataset=dev_ds,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    last = get_last_checkpoint(out_dir)
    print('Checkpoint bulundu, devam:' if last else 'Sıfırdan eğitim.', last or '')
    trainer.train(resume_from_checkpoint=last)   # oturum kapanırsa kaldığı yerden devam eder
    return trainer

trainer = egit(train_ds, dev_ds, CKPT_DIR)
trainer.save_model(MODEL_DIR); tok.save_pretrained(MODEL_DIR)
print('Model kaydedildi:', MODEL_DIR)

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sıfırdan eğitim. 


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.377833,0.366448,0.865314,0.848960
2,0.307415,0.381106,0.865314,0.849642
3,0.163856,0.435316,0.857472,0.842887
4,0.104854,0.505371,0.860240,0.845116


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model kaydedildi: /content/drive/MyDrive/cse654_absa/berturk_final


In [ ]:
def tahmin_ekle(trainer, ds, dfp):
    out = dfp.copy()
    out['pred']  = [id2label[i] for i in trainer.predict(ds).predictions.argmax(-1)]
    out['dogru'] = (out.pred == out.label).astype(int)
    return out

def skor(d):
    return pd.Series({'cumle': d.id.nunique(), 'cift': len(d),
                      'accuracy': accuracy_score(d.label, d.pred),
                      'macro_f1': f1_score(d.label, d.pred, labels=LABELS, average='macro', zero_division=0),
                      'cumle_tam_dogru': d.groupby('id').dogru.min().mean()})

def rapor_olustur(t):
    r = t.groupby('grup').apply(skor)
    r.loc['TÜMÜ'] = skor(t)
    return r.loc[['TÜMÜ', 'tek_hedefli', 'ayni_duygulu', 'notrle_karisik', 'zit_duygulu']].round(4)

test_pred = tahmin_ekle(trainer, test_ds, test_df)
rapor = rapor_olustur(test_pred)
print('Grup bazlı test sonuçları (cumle_tam_dogru = cümledeki bütün target\'lar doğru)')
display(rapor)

print(classification_report(test_pred.label, test_pred.pred, labels=LABELS, digits=4))
display(pd.DataFrame(confusion_matrix(test_pred.label, test_pred.pred, labels=LABELS),
                     index=['gerçek_' + l for l in LABELS], columns=['tahmin_' + l for l in LABELS]))

Grup bazlı test sonuçları (cumle_tam_dogru = cümledeki bütün target'lar doğru)


/tmp/ipykernel_729/714448838.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  r = t.groupby('grup').apply(skor)


,cumle,cift,accuracy,macro_f1,cumle_tam_dogru
grup,,,,,
TÜMÜ,2001.0,5407.0,0.8781,0.8598,0.7601
tek_hedefli,159.0,159.0,0.9245,0.9195,0.9245
ayni_duygulu,1003.0,2818.0,0.9344,0.9197,0.8734
notrle_karisik,447.0,1257.0,0.7542,0.7577,0.4832
zit_duygulu,392.0,1173.0,0.8696,0.7128,0.7194


              precision    recall  f1-score   support

    negative     0.8802    0.9319    0.9053      1806
     neutral     0.8226    0.7107    0.7625      1220
    positive     0.9005    0.9231    0.9117      2381

    accuracy                         0.8781      5407
   macro avg     0.8678    0.8552    0.8598      5407
weighted avg     0.8761    0.8781    0.8759      5407



,tahmin_negative,tahmin_neutral,tahmin_positive
gerçek_negative,1683,81,42
gerçek_neutral,152,867,201
gerçek_positive,77,106,2198


In [ ]:
# 1) Uzunluk eşlenmiş karşılaştırma: aynı duygulu grubun skorunu zıt duygulu grubun uzunluk dağılımına göre ağırlıklandır
zit, ayni = test_pred[test_pred.grup == 'zit_duygulu'], test_pred[test_pred.grup == 'ayni_duygulu']
kesim = np.unique(np.percentile(zit.kelime, [0, 20, 40, 60, 80, 100]))
kesim[0], kesim[-1] = 0, np.inf
zit_b, ayni_b = pd.cut(zit.kelime, kesim), pd.cut(ayni.kelime, kesim)
agirlik = zit_b.value_counts(normalize=True)
ayni_acc = ayni.groupby(ayni_b, observed=False).dogru.mean()
uzunluk = pd.DataFrame({
    'accuracy': [zit.dogru.mean(), ayni.dogru.mean(), (ayni_acc * agirlik).sum()]},
    index=['zıt duygulu', 'aynı duygulu (ham)', 'aynı duygulu (uzunluk eşlenmiş)']).round(4)
print('Uzunluk eşlenmiş karşılaştırma'); display(uzunluk)

# 2) Bağlaç (ama/fakat/ancak...) var mı yok mu
print('Bağlaç kırılımı'); display(test_pred.groupby(['grup', 'baglac']).dogru.agg(['size', 'mean']).round(4))

# 3) Zıt duygulu cümlelerdeki hatalar: model diğer target'ın duygusunu mu kopyalıyor?
def kopya_mi(r):
    digerleri = set(test_pred[(test_pred.id == r.id) & (test_pred.sira != r.sira)].label)
    return int(r.pred in digerleri)
zit_hata = zit[zit.dogru == 0].copy()
zit_hata['diger_target_kopyasi'] = zit_hata.apply(kopya_mi, axis=1)
print(f'Zıt duygulu hatalar: {len(zit_hata)} | diğer target\'ın duygusunu kopyalayan: %{100 * zit_hata.diger_target_kopyasi.mean():.1f}')
display(zit_hata[['id', 'text', 'target', 'label', 'pred']].head(15))

test_pred.to_csv(f'{SONUC_DIR}/{MODEL_KISA}_test_tahminleri.csv', index=False)
rapor.to_csv(f'{SONUC_DIR}/{MODEL_KISA}_grup_raporu.csv')
uzunluk.to_csv(f'{SONUC_DIR}/{MODEL_KISA}_uzunluk_eslenmis.csv')
print('Kaydedildi:', SONUC_DIR)

Uzunluk eşlenmiş karşılaştırma


,accuracy
zıt duygulu,0.8696
aynı duygulu (ham),0.9344
aynı duygulu (uzunluk eşlenmiş),0.9339


Bağlaç kırılımı


size    mean
grup           baglac              
ayni_duygulu   0       2595  0.9441
               1        223  0.8206
notrle_karisik 0        852  0.7394
               1        405  0.7852
tek_hedefli    0        143  0.9441
               1         16  0.7500
zit_duygulu    0        412  0.8083
               1        761  0.9028

Zıt duygulu hatalar: 153 | diğer target'ın duygusunu kopyalayan: %72.5


,id,text,target,label,pred
127,264,En alt katta kaldık Kaldığımız odadaki cam çok...,katta,neutral,negative
136,272,Eğitim için kullanışlı bir cihaz fakat bazı ek...,Kutu içeriğinde,neutral,negative
140,285,Fiyat güzel satıcı Amazon diye aldık kesekağıd...,satıcı Amazon,neutral,negative
158,327,Genel olarak TV izlemek kitap okumak ve not tu...,çektiği fotoğraflarda,neutral,positive
260,485,Kameraları güzel çevrimiçi toplantı ve belge t...,tipC kablo,neutral,negative
301,541,Konumu görüldüğü üzere gayet iyiydi Kahvaltısı...,Kahvaltısı,neutral,positive
371,619,Müzik seviyesi yeterliydi ancak bazen biraz da...,Müzik seviyesi,positive,neutral
373,619,Müzik seviyesi yeterliydi ancak bazen biraz da...,seviyede,negative,neutral
402,652,Odalar gayet güzel ve temiz Doğanın içinde s...,kahvaltı,negative,positive
446,695,Orijinal ürün lakin Apple kime yaptırıyorsa ar...,Orijinal ürün,positive,negative


Kaydedildi: /content/drive/MyDrive/cse654_absa/sonuclar


In [ ]:
model = trainer.model.eval()
def tahmin(cumle, hedef):
    enc = tok(cumle, hedef, return_tensors='pt', truncation='only_first', max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits, -1)[0].cpu().numpy()
    return {id2label[i]: round(float(x), 3) for i, x in enumerate(p)}

for h in ['konumu', 'oda']:
    print(h, '→', tahmin('Otelin konumu iyi ancak oda küçük', h))

konumu → {'negative': 0.003, 'neutral': 0.005, 'positive': 0.992}
oda → {'negative': 0.969, 'neutral': 0.026, 'positive': 0.006}


In [ ]:
if RUN_ABLATION:   # target vermeden, sadece cümle
    tr_a, dev_a, te_a = (hazirla(d, with_target=False) for d in (train_df, dev_df, test_df))
    trainer_a = egit(tr_a, dev_a, f'{PROJE}/{MODEL_KISA}_ablation_checkpoints')
    rapor_a = rapor_olustur(tahmin_ekle(trainer_a, te_a, test_df))
    karsi = pd.concat({'cümle+target': rapor[['accuracy', 'macro_f1']], 'sadece cümle': rapor_a[['accuracy', 'macro_f1']]}, axis=1)
    display(karsi); karsi.to_csv(f'{SONUC_DIR}/{MODEL_KISA}_ablation.csv')
else:
    print('Ablation kapalı.')

Ablation kapalı.
